## Imports

In [20]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_curve,
)
from sklearn.model_selection import (
    FixedThresholdClassifier,
    GroupShuffleSplit,
    GridSearchCV,
    GroupKFold,
)
from sklearn.pipeline import Pipeline
from itertools import combinations
import numpy as np

### Load full dataset

In [5]:
source = pd.read_json("python_dataset.jsonl", lines=True)

### Prepare Dataset

In [14]:
df = source

df = df.sample(1000, random_state=42)
df = df.reset_index(drop=True)

df = df.drop(columns="docstring")

# Convert df from wide format to long format. Originally it was idx, human_code, gpt_code, qwen_code ...
# But i need seperate rows per code snippet. With melt I transform it to idx, code, source
df = pd.melt(
    df,
    id_vars="hm_index",
    value_vars=["human_code", "chatgpt_code", "dsc_code", "qwen_code"],
    var_name="source",
    value_name="code",
)

df = df.dropna(axis=0)

# Add new column thats 0 or 1 depending on source. Human-written is 0, AI-generated is 1
df["ai_generated"] = df["source"].ne("human_code").astype(int)

df = df.rename(columns={"hm_index": "task_id"})

# Simplify task_id values and sort df
df["task_id"] = df.groupby("task_id").ngroup() + 1
df = df.sort_values(by=["task_id"])

# Move colum order so source comes last
df = df[["task_id", "code", "ai_generated", "source"]]

### Split df into training and test

In [15]:
# Split code into input X and ouput y
all_task_ids = df["task_id"]
X = df["code"]
y = df["ai_generated"]

# The df has multiple rows per coding task. e.g. task 1 has 4 rows (human-, gpt-, qwen- and dsc- code snippets)
# with GroupShuffleShift I split the the rows but dont put rows with the same task_id into train AND test
gss_test = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)

train_val_idx, test_idx = next(gss_test.split(X, y, groups=all_task_ids))
test_X, test_y = X.iloc[test_idx], y.iloc[test_idx]


# train val split
gss_val = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)

train_val_df = df.iloc[train_val_idx]
X_tv = train_val_df["code"]
y_tv = train_val_df["ai_generated"]
train_val_task_ids = train_val_df["task_id"]

train_idx, val_idx = next(gss_val.split(X_tv, y_tv, groups=train_val_task_ids))

train_X, train_y = X_tv.iloc[train_idx], y_tv.iloc[train_idx]
val_X, val_y = X_tv.iloc[val_idx], y_tv.iloc[val_idx]

# Check if there are any overlaps in train/test/val
train_X_set, test_X_set, val_X_set = map(set, (train_X, test_X, val_X))


train_task_ids = train_val_task_ids.iloc[train_idx]
val_task_ids = train_val_task_ids.iloc[val_idx]
test_task_ids = all_task_ids.iloc[test_idx]


train_task_ids_set, val_task_ids_set, test_task_ids_set = map(
    set, (train_task_ids, val_task_ids, test_task_ids)
)
splits = (train_task_ids_set, val_task_ids_set, test_task_ids_set)

is_clean = all(a.isdisjoint(b) for a, b in combinations(splits, 2))
print("There is no overlap between splits") if is_clean else print("OVERLAP WARNING")

There is no overlap between splits


### Build Pipeline and optimize it for f1_macro score

In [ ]:
# Init a Pipeline with 3 steps:
# 1. Count vectorizer to make a count vector for each row in the provided data (only the code)
# 2. TfidfTransformer takes count vectors from previous step and produces tf-idf vectors
# 3. Logreg as our model

pipeline = Pipeline(
    steps=[
        ("cv", CountVectorizer()),
        ("tfidf", TfidfTransformer()),
        ("logreg", LogisticRegression(max_iter=1000, random_state=42)),
    ]
)

# evtl fbeta score mit tiefem beta machen. je höher fbeta desto stärker wird recall belohnt

param_grid = {"logreg__C": np.geomspace(500, 2000, num=30).tolist()}
print(param_grid)

group_k_fold = GroupKFold(n_splits=5)

grid_search_cv = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    refit=True,
    cv=group_k_fold,
    scoring="f1_macro",
    verbose=2,
    n_jobs=-1,
)

grid_search_cv.fit(train_X, train_y, groups=train_task_ids)

best_pipeline = grid_search_cv.best_estimator_
print(best_pipeline.get_params)


{'logreg__C': [100.0, 166.81005372000593, 278.2559402207126, 464.15888336127773, 774.263682681127, 1291.549665014884, 2154.4346900318824, 3593.813663804626, 5994.8425031894085, 10000.0]}
Fitting 5 folds for each of 10 candidates, totalling 50 fits
[CV] END ....................................logreg__C=100.0; total time=   0.1s
[CV] END ....................................logreg__C=100.0; total time=   0.1s
[CV] END ....................................logreg__C=100.0; total time=   0.2s
[CV] END .......................logreg__C=166.81005372000593; total time=   0.2s
[CV] END ....................................logreg__C=100.0; total time=   0.2s
[CV] END .......................logreg__C=166.81005372000593; total time=   0.1s
[CV] END ....................................logreg__C=100.0; total time=   0.2s
[CV] END .......................logreg__C=166.81005372000593; total time=   0.1s
[CV] END .......................logreg__C=166.81005372000593; total time=   0.2s
[CV] END ..............

### Treshold tuning using the validation set

In [2]:
pred_proba = best_pipeline.predict_proba(val_X)

positive_class_probs = pred_proba[:, 1]

fpr, tpr, threshold = roc_curve(val_y, positive_class_probs)

for f, t, thr in zip(fpr, tpr, threshold):
    if f == 0.05:
        print(f"Threshhold: {thr:.5f} -> FPR: {f:.5f} -> TPR: {t:.5f}")
        optimal_threshold = threshold

# optimal threshhold at about Threshhold: 0.895 -> FPR: 0.050 -> TPR: 0.735


NameError: name 'best_pipeline' is not defined